# GPU Hello World

### Task 3

1. Make a NumPy matrix
2. Install, import and initialize PyCUDA
3. Make a GPU matrix
4. Copy the NumPy matrix data to the GPU
5. Write a GPU *kernel* that does some kind of computation on an input matrix
6. Copy the result back to the CPU
7. Print and plot both matrices

### Solution 3

<-- see `hello_world_pycuda_solution.ipynb`

In [1]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.0 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=19d9c4b23a7a18402ad8410018479ce65d4d1429551fea7dd02fbdea40eeeeee
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [7]:
import numpy as np
from pycuda.compiler import SourceModule
import pycuda.driver as cuda

N = 1024
h_matrix = np.random.randn(N, N).astype(np.float32)
h_result = np.empty_like(h_matrix)

THR_PER_BLOCK = 32
num_blocks = int(np.ceil(N / THR_PER_BLOCK))

mod = SourceModule("""
__global__ void transform_matrix(float *dest, float *src, int n)
{
    // Localizar la posición global del hilo
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    // IMPORTANTE: En C++, el acceso a matriz 2D plana es: y * ancho + x
    if (x < n && y < n)
    {
        int idx = y * n + x;
        dest[idx] = src[idx] * 2.0f + 10.0f;
    }
}
""")

transform_matrix = mod.get_function("transform_matrix")

transform_matrix(
    cuda.Out(h_result),
    cuda.In(h_matrix),
    np.int32(N),
    block=(THR_PER_BLOCK,THR_PER_BLOCK,1),
    grid=(num_blocks,num_blocks)
)

print(h_matrix)
print(h_result)


CompileError: nvcc compilation of /tmp/tmpysxa7aib/kernel.cu failed
[command: nvcc --cubin -arch sm_75 -I/usr/local/lib/python3.12/dist-packages/pycuda/cuda kernel.cu]
[stderr:
kernel.cu(12): error: expression must have pointer-to-object type but it has type "float"
          dest[x][y] = src[x][y] * 2 + 10;
          ^

kernel.cu(12): error: expression must have pointer-to-object type but it has type "float"
          dest[x][y] = src[x][y] * 2 + 10;
                       ^

2 errors detected in the compilation of "kernel.cu".
]